# 10. Live CIFAR-10 classification from a USB webcam

Use the pretrained **CNV-W1A1** network on the PYNQ FPGA to classify successive camera frames. Designed for PYNQ-Z1/Z2 and a USB webcam such as the Logitech C270, using the existing BNN-PYNQ installation.

**Objectives:** capture frames continuously, prepare RGB input, run inference on the FPGA, and distinguish accelerator timing from application throughput.

The model assigns one of ten CIFAR-10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. It classifies the selected image region; it does not locate objects. A cup, face, or empty background still receives one of these labels. Predictions on camera images can be poor because CIFAR-10 contains small training images with different framing and backgrounds.

This supplementary lab uses Xilinx BNN-PYNQ's public Python API. Original project: https://github.com/Xilinx/BNN-PYNQ . Keep the repository's LICENSE and original notices.

## 1. Before running

1. Run the original `01_CNV-BNN_Cifar10.ipynb` successfully first.
2. Shut down the kernels of other BNN/camera notebooks. Connect the C270 to the **PYNQ board**, not the PC.
3. Run the cells below in order. No package reinstallation, model training, or Vivado build is needed.
4. The first live session stops after **30 seconds**. To stop sooner, use **Kernel → Interrupt** (the toolbar stop button).

The camera is opened only inside a `try/finally` block and released on normal completion, Python exceptions, or a handled keyboard interrupt. If a native camera/FPGA call itself hangs, interruption may be delayed; use **Kernel → Restart**, then shut down any other camera kernels before retrying.

In [ ]:
import contextlib
import io
import time
import cv2
import numpy as np
from PIL import Image as PILImage
from IPython.display import display, Image as NotebookImage
import bnn

# BNN-PYNQ uses Image.ANTIALIAS, renamed in newer Pillow releases.
if not hasattr(PILImage, "ANTIALIAS"):
    PILImage.ANTIALIAS = PILImage.Resampling.LANCZOS

print("OpenCV:", cv2.__version__)
print("BNN module:", bnn.__file__)

## 2. Configure the experiment

The central square (green outline) is classified by default. Put the object inside it and use a simple background. Set `CENTER_CROP = False` to classify the entire frame, resized to a square.

A 640 × 480 capture is a request; the camera may negotiate another resolution. There is no artificial FPS target. `MAX_SECONDS` limits the session, but cannot impose a timeout on a blocked native driver call.

In [ ]:
CAMERA_DEVICE = "/dev/video0"
CAPTURE_WIDTH = 640
CAPTURE_HEIGHT = 480
WARMUP_FRAMES = 5
MAX_SECONDS = 30.0
MAX_FRAMES = 300  # A second stop condition; the first limit reached wins.
CENTER_CROP = True
JPEG_QUALITY = 75
NETWORK = bnn.NETWORK_CNVW1A1
RUNTIME = bnn.RUNTIME_HW

assert MAX_SECONDS > 0 and MAX_FRAMES > 0
assert 1 <= JPEG_QUALITY <= 100

## 3. Load the pretrained classifier once

This downloads the matching overlay if needed and loads CIFAR-10 parameters. Keep this classifier for all frames; do not create it inside the loop. No `PL.reset()` is needed. To try another network later, restart the kernel and rerun the notebook with the new setting.

In [ ]:
if "cifar10" not in bnn.available_params(NETWORK):
    raise RuntimeError("CIFAR-10 parameters are missing for " + NETWORK)
classifier = bnn.CnvClassifier(NETWORK, "cifar10", RUNTIME)
print("Network:", NETWORK)
print("Runtime:", RUNTIME)
print("Classes:", ", ".join(classifier.classes))

## 4. Camera and inference helpers

Each selected region is explicitly converted **BGR → RGB** and resized to **32 × 32**. The existing classifier handles CIFAR binary formatting. The pixelated inset shows exactly this RGB input, enlarged for inspection.

Three different measurements are used:

- **Accelerator ms:** `classifier.usecPerImage / 1000`, reported by the BNN runtime. It is not a measurement of the complete camera application.
- **Classify call ms:** wall time of `classify_image`, including its image formatting, temporary file, and inference call.
- **Application FPS:** completed iterations divided by elapsed time, including frame reads, preprocessing, inference, annotation, JPEG encoding, and sending notebook updates. Browser rendering and network delivery are not measured. During the session the FPS label reports the preceding completed iterations; the final summary includes the final update.

No confidence percentage is displayed: the classifier's class rankings are not calibrated probabilities.

In [ ]:
def read_frame(cap):
    ok, frame = cap.read()
    if not ok or frame is None or frame.size == 0:
        raise RuntimeError(
            "No camera frame. Check the USB connection and shut down "
            "other camera notebook kernels."
        )
    return frame


def prepare_input(frame, center_crop):
    height, width = frame.shape[:2]
    if center_crop:
        side = min(height, width)
        x0, y0 = (width - side) // 2, (height - side) // 2
        region = frame[y0:y0 + side, x0:x0 + side]
        box = (x0, y0, side)
    else:
        region, box = frame, None
    rgb = cv2.cvtColor(region, cv2.COLOR_BGR2RGB)
    rgb32 = cv2.resize(rgb, (32, 32), interpolation=cv2.INTER_AREA)
    return PILImage.fromarray(rgb32), rgb32, box


def run_live(classifier):
    cap = None
    completed = 0
    accelerator_ms, call_ms = [], []
    start = None
    elapsed = 0.0
    interrupted = False
    try:
        cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
        if not cap.isOpened():
            raise RuntimeError("Cannot open " + str(CAMERA_DEVICE)
                               + ". Check the camera and other kernels.")
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAPTURE_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAPTURE_HEIGHT)
        # Best effort: not all V4L2 drivers support this setting.
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        for _ in range(WARMUP_FRAMES):
            read_frame(cap)
        print("Camera opened. Stop early with Kernel > Interrupt.")
        view = display(NotebookImage(data=cv2.imencode(
            ".jpg", np.zeros((48, 320, 3), dtype=np.uint8)
        )[1].tobytes()), display_id=True)
        start = time.perf_counter()
        while completed < MAX_FRAMES and time.perf_counter() - start < MAX_SECONDS:
            frame = read_frame(cap)
            image, rgb32, box = prepare_input(frame, CENTER_CROP)
            call_start = time.perf_counter()
            # Suppress the library's Python timing prints on each frame.
            with contextlib.redirect_stdout(io.StringIO()):
                class_id = int(classifier.classify_image(image))
            wall_ms = (time.perf_counter() - call_start) * 1000.0
            hardware_ms = float(classifier.usecPerImage) / 1000.0
            label = classifier.class_name(class_id)
            fps = completed / elapsed if elapsed > 0 else 0.0

            annotated = frame.copy()
            if box is not None:
                x, y, side = box
                cv2.rectangle(annotated, (x, y), (x + side - 1, y + side - 1),
                              (0, 255, 0), 2)
            # A fixed panel keeps text readable at different capture sizes.
            annotated = cv2.resize(annotated, (640, 480))
            panel = np.zeros((100, 640, 3), dtype=np.uint8)
            lines = [
                "Prediction: {} | {}".format(label, NETWORK),
                "Accelerator: {:.2f} ms | Classify call: {:.2f} ms".format(
                    hardware_ms, wall_ms),
                "Application: {:.2f} FPS | completed frames: {}".format(fps, completed),
            ]
            for i, text in enumerate(lines):
                cv2.putText(panel, text, (10, 24 + i * 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.53, (255, 255, 255), 1,
                            cv2.LINE_AA)
            inset = cv2.resize(cv2.cvtColor(rgb32, cv2.COLOR_RGB2BGR),
                               (128, 128), interpolation=cv2.INTER_NEAREST)
            annotated[342:470, 502:630] = inset
            cv2.putText(annotated, "32x32 input", (500, 335),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
            canvas = np.vstack((panel, annotated))
            ok, jpeg = cv2.imencode(".jpg", canvas,
                                    [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            if not ok:
                raise RuntimeError("JPEG encoding failed.")
            view.update(NotebookImage(data=jpeg.tobytes()))
            completed += 1
            accelerator_ms.append(hardware_ms)
            call_ms.append(wall_ms)
            elapsed = time.perf_counter() - start
    except KeyboardInterrupt:
        interrupted = True
        print("Session interrupted.")
    finally:
        if cap is not None:
            cap.release()
        print("Camera released.")

    result = {
        "frames": completed,
        "elapsed_s": elapsed,
        "application_fps": completed / elapsed if elapsed > 0 else 0.0,
        "mean_accelerator_ms": float(np.mean(accelerator_ms)) if completed else None,
        "mean_classify_call_ms": float(np.mean(call_ms)) if completed else None,
        "interrupted": interrupted,
    }
    print("Completed frames: {} | measured loop time: {:.2f} s | {:.2f} FPS".format(
        completed, elapsed, result["application_fps"]))
    if completed:
        print("Mean accelerator: {:.2f} ms | mean classify call: {:.2f} ms".format(
            result["mean_accelerator_ms"], result["mean_classify_call_ms"]))
    return result

## 5. Run the live session

Run this cell and move the object slowly inside the green square. The cell remains busy (`[*]`) during capture; the image should update. The final camera frame remains visible after stopping. Rerun **only this cell** for another session; the camera is reopened automatically.

The 30-second limit starts after camera warmup. A slow frame or inference may make the session finish slightly later. Avoid running other FPGA overlays during this session.

In [ ]:
session_stats = run_live(classifier)

## 6. Student experiments

1. **Framing and background:** compare `CENTER_CROP = True` and `False`. Use the same object, first with a plain background, then with clutter. Record whether the prediction stays stable. Inspect the 32 × 32 inset.
2. **Application cost:** record accelerator ms, classify-call ms, and application FPS. Explain why `1000 / accelerator_ms` is higher than the measured application FPS. Try `JPEG_QUALITY = 50`; do not assume that it always improves FPS.
3. **Known and unknown categories:** try a printed CIFAR-10 object (for example a car), then a cup or phone. Explain why the classifier still returns a known category. There is no trained “unknown” class.

Optional: restart the kernel, set `NETWORK = bnn.NETWORK_CNVW1A2` or `bnn.NETWORK_CNVW2A2`, and rerun in the same conditions. Compare observed speed and predictions. A few webcam frames do not establish dataset accuracy; use the Testset notebook for that.

| Network | Crop | Accelerator ms | Classify call ms | Application FPS | Observations |
|---|---|---:|---:|---:|---|
| W1A1 | Central square | | | | |
| W1A1 | Entire frame | | | | |

## Troubleshooting

- **Camera cannot open / no frame:** check the USB connection and shut down the kernels of all other camera notebooks. Verify `/dev/video0` is the correct device; change `CAMERA_DEVICE` if necessary.
- **Cell stays busy but frames update:** expected during the live loop. Use Kernel → Interrupt to stop early.
- **No updates and interrupt does not work:** a native call may be blocked. Restart the kernel and check the USB camera. Do not start another capture in a second notebook while the first kernel is still running.
- **`import bnn` or overlay loading fails:** first resolve the same failure in the original BNN example. This lab reuses the installed package and bitstreams.
- **Labels fluctuate:** keep the target large, centered, and well lit; use a plain background. Classification of arbitrary real scenes remains limited by the pretrained CIFAR-10 model.

**Validation status:** notebook structure and code are checked off-board; camera/FPGA execution and actual FPS require testing on the PYNQ board.